In [21]:
import ConnectionConfig as cc
from delta import DeltaTable
from pyspark.sql.functions import col, hour, explode, from_unixtime, unix_timestamp
cc.setupEnvironment()

In [22]:
spark = cc.startLocalCluster("S1S2FactRide")
spark.getActiveSession()

In [23]:
#Extract factRide1 factRide2
fact_ride_s2 = spark.read.format("delta").load("spark-warehouse/factRidesS2")
fact_ride_s1 = spark.read.format("delta").load("spark-warehouse/factRideS1")

In [24]:
#Transform dimWeather and dimDate
fact_ride_s1.createOrReplaceTempView("factRideS1")
fact_ride_s2.createOrReplaceTempView("factRideS2")
# dim_customer.createOrReplaceTempView("dimCustomer")
# dim_locks.createOrReplaceTempView("dimLock")

In [25]:
fact_rides_S1_S2 = spark.sql("""
SELECT
        frs1.rideSk, frs1.dateSK, frs2.startLockSK, frs2.endLockSK, frs2.customerSK, frs1.weatherSK, 
        frs1.ride_duration, frs2.rideDistance
        FROM factRideS1 frs1
        JOIN factRideS2 frs2 ON frs1.rideSK = frs2.rideSK
""")

fact_rides_S1_S2.show(20)

+------+------+-----------+---------+----------+---------+-------------+--------------------+
|rideSk|dateSK|startLockSK|endLockSK|customerSK|weatherSK|ride_duration|        rideDistance|
+------+------+-----------+---------+----------+---------+-------------+--------------------+
|     2|  2455|         49|     1003|8589981203|        3|    -94608000|2.002955676810820000|
|     4|  3916|       5976|      696|8589956306|        4|          304|2.859825631258500000|
|     5|  3916|       5976|      696|8589956305|        4|          395|2.859825631258500000|
|     8|  3916|       1759|       67|8589971140|        4|          677|3.872382206062740000|
|    12|  3916|       4081|     6047|8589969232|        4|          269|3.018637650324220000|
|    15|  3916|       5615|     5813|8589935589|        4|         1143|0.940218180145754000|
|    19|  3916|       4261|     1219|8589959948|        4|          647|4.283761945729960000|
|    23|  3916|        139|     2605|8589989032|        4|  

In [26]:
# Define path for the Delta table
fact_ride_path = "spark-warehouse/fact_ride_s1_s2"

fact_rides_S1_S2.write.format("delta").mode("overwrite").save(fact_ride_path)